# Pipeline de production

Le pipeline de production reprend la séquence de pré-production.
Le déploiement de l'API cible Kubernetes, sur la branche `master` des deux dépôts.
Le déclencheur de `purchase_predict` lance le build de `purchase_predict_api` après l'entraînement.
Quatre déclencheurs couvrent les deux projets et les deux environnements.
Les journaux Gunicorn remontent dans Cloud Logging.


## Branche master

Dans le dépôt Kedro `purchase_predict`, reprendre `master` et y fusionner `staging`. Les étapes CI et CD restent celles de la pré-production. Seule l'étape `API Deploy` change : elle clone `purchase_predict_api` depuis Cloud Source Repositories, se place sur `$BRANCH_NAME`, puis soumet le `cloudbuild.yaml` de cette branche. Pour le déclencheur de production, `BRANCH_NAME` vaut `master`.

Le secret `GITHUB_TOKEN` n'est plus référencé. Le secret `cloud-build` reste nécessaire aux étapes CI et CD. Le fichier conserve la machine `N1_HIGHCPU_8` et un timeout de 1200 s, au-delà de la limite par défaut de 10 minutes.


## Déclencheurs

Dupliquer `build-purchase-predict-staging` en conservant ses substitutions, dont `_MLFLOW_SERVER`, et viser la branche `master`. Au total, quatre déclencheurs : `purchase_predict` et `purchase_predict_api`, chacun en `staging` et en `master`.

L'étape `API Deploy` soumet un second build. Il construit l'image, la pousse, rend les manifestes et met à jour le Deployment dans `api-purchase`. Le tag d'image du Deployment correspond au `SHORT_SHA` de ce build.


## Journaux

Cloud Logging reçoit les sorties de l'API. Quatre flux identiques correspondent aux deux workers Gunicorn de chacun des deux pods.
